In [2]:
# -------------------------------------------------------
# CELL 1 - Install dependencies and config
# -------------------------------------------------------
%pip install google-cloud-bigquery pandas-gbq

import os
from datetime import datetime, timezone
import uuid
from pyspark.sql import SparkSession

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
GCP_PROJECT_ID = "cyberpulse-490516"
BQ_DATASET = "soc_internal_logs"
BQ_TABLE = "firewall_logs"
PIPELINE_RUN_ID = str(uuid.uuid4())
SOURCE_SYSTEM = "bigquery"

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Timestamp: {datetime.now(timezone.utc)}")
print(f"Source: {GCP_PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}")

StatementMeta(, 9e809655-1ebf-49cb-9c25-34cee6d67d84, 9, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.3/262.3 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.3/173.3 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 115.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os
from datetime import datetime, timezone
import uuid

GCP_PROJECT_ID = "cyberpulse-490516"
BQ_DATASET = "soc_internal_logs"
BQ_TABLE = "firewall_logs"
PIPELINE_RUN_ID = str(uuid.uuid4())
SOURCE_SYSTEM = "bigquery"

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")
print(f"Timestamp: {datetime.now(timezone.utc)}")
print(f"Source: {GCP_PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}")

StatementMeta(, 9e809655-1ebf-49cb-9c25-34cee6d67d84, 11, Finished, Available, Finished, False)

Pipeline Run ID: e747f04f-a401-4668-a5b0-b353098406b8
Timestamp: 2026-05-10 12:32:06.164587+00:00
Source: cyberpulse-490516.soc_internal_logs.firewall_logs


In [5]:
# -------------------------------------------------------
# CELL 2 - Read from BigQuery
# -------------------------------------------------------
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import os

# -------------------------------------------------------
# SECURITY NOTE: In production use Key Vault
#
# -------------------------------------------------------
KEY_FILE_PATH = "SET_PATH_TO_GCP_SERVICE_ACCOUNT_KEY"

credentials = service_account.Credentials.from_service_account_file(
    KEY_FILE_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bq_client = bigquery.Client(
    project=GCP_PROJECT_ID,
    credentials=credentials
)

query = f"""
    SELECT 
        log_id,
        timestamp,
        internal_ip,
        destination,
        destination_type,
        action,
        bytes_transferred,
        department,
        severity
    FROM `{GCP_PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}`
"""

print("Fetching data from BigQuery...")
df_pandas = bq_client.query(query).to_dataframe()
print(f"Rows fetched: {len(df_pandas)}")
print(f"Columns: {list(df_pandas.columns)}")

StatementMeta(, 9e809655-1ebf-49cb-9c25-34cee6d67d84, 13, Finished, Available, Finished, False)

Fetching data from BigQuery...
Rows fetched: 1000
Columns: ['log_id', 'timestamp', 'internal_ip', 'destination', 'destination_type', 'action', 'bytes_transferred', 'department', 'severity']


In [6]:
# -------------------------------------------------------
# CELL 3 - Convert to Spark and add Bronze technical columns
# -------------------------------------------------------
from pyspark.sql.functions import lit
from datetime import datetime, timezone

ingested_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

df_firewall = spark.createDataFrame(df_pandas)

df_firewall = (df_firewall
    .withColumn("ingested_at", lit(ingested_at))
    .withColumn("source_system", lit(SOURCE_SYSTEM))
    .withColumn("pipeline_run_id", lit(PIPELINE_RUN_ID))
)

print(f"Spark DataFrame: {df_firewall.count()} rows, {len(df_firewall.columns)} columns")
df_firewall.printSchema()

StatementMeta(, 9e809655-1ebf-49cb-9c25-34cee6d67d84, 14, Finished, Available, Finished, False)

Spark DataFrame: 1000 rows, 12 columns
root
 |-- log_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- internal_ip: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- destination_type: string (nullable = true)
 |-- action: string (nullable = true)
 |-- bytes_transferred: long (nullable = true)
 |-- department: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- ingested_at: string (nullable = false)
 |-- source_system: string (nullable = false)
 |-- pipeline_run_id: string (nullable = false)



In [7]:
# -------------------------------------------------------
# CELL 4 - Write to Bronze Delta table: bronze_firewall_logs
# -------------------------------------------------------
bronze_table_firewall = "bronze_firewall_logs"

(df_firewall.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_table_firewall)
)

count = spark.sql(f"SELECT COUNT(*) as total FROM {bronze_table_firewall}").collect()[0]["total"]
print(f"bronze_firewall_logs: {count} rows written successfully")

StatementMeta(, 9e809655-1ebf-49cb-9c25-34cee6d67d84, 15, Finished, Available, Finished, False)

bronze_firewall_logs: 1000 rows written successfully


In [8]:
# -------------------------------------------------------
# CELL 5 - Pipeline logging
# -------------------------------------------------------
log_row = [{
    "pipeline_run_id": PIPELINE_RUN_ID,
    "pipeline_name": "nb_bronze_bigquery_ingestion",
    "source_system": SOURCE_SYSTEM,
    "table_name": "bronze_firewall_logs",
    "rows_loaded_pulses": 0,
    "rows_loaded_indicators": df_firewall.count(),
    "status": "SUCCESS",
    "run_timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    "notes": "Initial Bronze load - firewall logs from BigQuery"
}]

df_log = spark.createDataFrame(log_row)

(df_log.write
    .format("delta")
    .mode("append")
    .saveAsTable("pipeline_log")
)

print(f"Pipeline log written for run: {PIPELINE_RUN_ID}")
spark.sql("SELECT * FROM pipeline_log").show(truncate=50)

StatementMeta(, 9e809655-1ebf-49cb-9c25-34cee6d67d84, 16, Finished, Available, Finished, False)

Pipeline log written for run: e747f04f-a401-4668-a5b0-b353098406b8
+--------------------------------------------------+----------------------------+------------------------------------+----------------------+------------------+-------------------+-------------+-------+----------------------------------------+
|                                             notes|               pipeline_name|                     pipeline_run_id|rows_loaded_indicators|rows_loaded_pulses|      run_timestamp|source_system| status|                              table_name|
+--------------------------------------------------+----------------------------+------------------------------------+----------------------+------------------+-------------------+-------------+-------+----------------------------------------+
|Initial Bronze load - 5 pages OTX subscribed pu...|     nb_bronze_otx_ingestion|23e36d74-44ad-413a-b858-cb654a0cc10f|                  6881|               250|2026-05-03 12:46:42|      otx_api|SUCCESS